In [14]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [3]:
df = pd.read_excel('yahoo_news_21jul.xlsx', index_col=0)

In [4]:
df2 = df[df['source'] != 'http://www.macaushimbun.com'].reset_index (drop = True).copy() #убираем статьи про Макао, нас они не интересуют

In [5]:
df2['comments'] = df2['comments'].astype(int)

In [9]:
df2['category'] = pd.cut(df2['comments'], 
                         bins=[-1, 15, np.inf], 
                         labels=[0, 1])

print(df2['category'].value_counts())

category
0    1400
1     371
Name: count, dtype: int64


In [10]:
df3 = df2 [['category', 'text']]

In [13]:
train_df, test_df = train_test_split(
    df3,
    test_size=0.2,
    random_state=42,
    stratify=df3['category'],
    shuffle = True
)

In [ ]:
def set_seed(seed_value=42):
    '''для воспроизводимости результата'''
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [ ]:
set_seed(42)

In [ ]:
#дообучаем модель BERT
model_name = "cl-tohoku/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(model_name, meccan_backend="sudachi")

In [ ]:
def tokenize_bert(texts):
    return tokenizer(
        texts.tolist(), 
        truncation=True, 
        padding=True, 
        max_length=256
    )

In [ ]:
train_encodings = tokenize_bert(train_df['text'])
test_encodings = tokenize_bert(test_df['text'])

In [ ]:
class JapaneseNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_bert = JapaneseNewsDataset(train_encodings, train_df['category'])
test_bert = JapaneseNewsDataset(test_encodings, test_df['category'])

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    f1_macro = f1_score(labels, predictions, average='macro')
    return {'f1_macro': f1_macro}

In [ ]:
model_bert = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
training_args = TrainingArguments(
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.1, #регуляризация для защиты от переобучения
    learning_rate=1e-5, #пониженный learning rate для защиты от переобучения
    logging_steps=10,
    seed=42,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available()
)

In [ ]:
trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_bert,
    eval_dataset=test_bert,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
raw_predictions = trainer.predict(test_bert)
y_bert = np.argmax(raw_predictions.predictions, axis=1)

In [ ]:
print(classification_report(test_df['category'], y_bert))

In [ ]:
cm = confusion_matrix (test_df['category'], y_bert)
cm_display = ConfusionMatrixDisplay (cm).plot()

In [15]:
classic_results = pd.read_excel('classic_models_results.xlsx')

In [ ]:
results_report = []

In [ ]:
def scores (model, y_test, y_pred):
  report_dict = classification_report(y_test, y_pred, output_dict=True)
  accuracy = report_dict['accuracy']
  f1_macro = report_dict['macro avg']['f1-score'] #больше всего нас интересует эта метрика, тк есть дисбаланс классов
  precision = report_dict ['macro avg']['precision']
  recall = report_dict ['macro avg']['recall']
  results_report.append ([model, accuracy, f1_macro, precision, recall])
  return results_report

In [ ]:
scores ('BERT', test_df['category'], y_bert)

In [ ]:
results_table = pd.DataFrame (results_report, columns=['model', 'accuracy', 'f1_macro', 'precision', 'recall'])
final_results = pd.concat ([classic_results, results_table], axis = 0, ignore_index=True)
final_results.sort_values(by='f1_macro', ascending=False)